# Required Setup

### Estimated setup time: 5 minutes

This notebook prepares your Databricks environment for the Graph Augmented AI Workshop.
It creates the required catalog, schema, volume, copies data files, and configures
Neo4j credentials.

**Run this notebook once before starting any labs.**

### Importing the Labs

Import only the **notebook files** (`.py`) and `Includes/_lib/` folder into your workspace.
Data files (CSV, HTML, embeddings) are downloaded automatically from GitHub during setup.

### Prerequisites

- A **Dedicated** compute cluster with the Neo4j Spark Connector installed:
  - Access mode: **Dedicated** (required for the Spark Connector)
  - Runtime: 13.3 LTS or higher
  - Maven library: `org.neo4j:neo4j-connector-apache-spark_2.12:5.3.1_for_spark_3`
- A running **Neo4j** instance (Aura or self-hosted) with connection details ready
- Permission to **create a catalog** in your workspace (or an existing catalog you can use)

## Step 1: Enter Your Neo4j Connection Details

Fill in the widgets at the top of this notebook with your Neo4j connection information.
These values will be stored as Databricks secrets so subsequent notebooks can use them.

### Where Are My Neo4j Credentials Stored?

Your Neo4j connection details are saved as Databricks secrets in the scope
**`neo4j-graph-enrichment-creds`** with keys `url`, `username`, `password`, and `volume_path`.
All subsequent lab notebooks read from this scope automatically.

To change the scope name, edit `CONFIG["secrets"]["scope_name"]` in
`Includes/config.ipynb` before running setup. If you've already run setup under a
different scope name, re-run this notebook to create the new scope and store the credentials again.

In [ ]:
dbutils.widgets.text("neo4j_url", "", "Neo4j URI (e.g. neo4j+s://xxx.databases.neo4j.io)")
dbutils.widgets.text("neo4j_username", "neo4j", "Neo4j Username")
dbutils.widgets.text("neo4j_password", "", "Neo4j Password")

## Step 2: Run the Setup

The cell below will:
1. Create a catalog and schema based on your username
2. Download all CSV, HTML, and embedding data files from GitHub into your volume
3. Store your Neo4j credentials as Databricks secrets
4. Verify the Neo4j connection

Review the output to confirm everything succeeded.

In [ ]:
# Get widget values
neo4j_url = dbutils.widgets.get("neo4j_url")
neo4j_username = dbutils.widgets.get("neo4j_username")
neo4j_password = dbutils.widgets.get("neo4j_password")

if not neo4j_url or not neo4j_password:
    raise ValueError(
        "Please fill in the Neo4j URI and Password widgets at the top of this notebook before running."
    )

In [ ]:
%run ./Includes/config

In [ ]:
%run ./Includes/_lib/setup_orchestrator

In [ ]:
catalog_config = CONFIG["catalog"]
secrets_config = CONFIG["secrets"]
github_config = CONFIG["github"]

print("Configuration loaded:")
print(f"  Catalog prefix: {catalog_config['prefix']}")
print(f"  Schema:         {catalog_config['schema_name']}")
print(f"  Volume:         {catalog_config['volume_name']}")
print(f"  Secret scope:   {secrets_config['scope_name']}")
print(f"  Data source:    github.com/{github_config['repo']} ({github_config['branch']})")

In [ ]:
# Step 1: Create catalog, schema, and volume
username = get_username()
catalog_name = derive_catalog_name(catalog_config["prefix"], username)

catalog_info = setup_catalog_and_schema(
    catalog_name=catalog_name,
    schema_name=catalog_config["schema_name"],
    volume_name=catalog_config["volume_name"],
)

In [ ]:
# Step 2: Download data files from GitHub to volume
file_counts = download_data_files(
    volume_path=catalog_info["volume_path"],
    github_repo=github_config["repo"],
    github_branch=github_config["branch"],
    data_path=github_config["data_path"],
)

In [ ]:
# Step 3: Store Neo4j secrets
setup_neo4j_secrets(
    scope_name=secrets_config["scope_name"],
    neo4j_url=neo4j_url,
    neo4j_username=neo4j_username,
    neo4j_password=neo4j_password,
    volume_path=catalog_info["volume_path"],
)

In [ ]:
# Step 4: Verify Neo4j connection
neo4j_connected = verify_neo4j_connection(neo4j_url, neo4j_username, neo4j_password)

In [ ]:
# Print summary
print_summary({
    "catalog": catalog_info["catalog"],
    "schema": catalog_info["schema"],
    "volume": catalog_info["volume"],
    "volume_path": catalog_info["volume_path"],
    "neo4j_url": neo4j_url,
    "scope_name": secrets_config["scope_name"],
    "file_counts": file_counts,
    "neo4j_connected": neo4j_connected,
})

## Next Steps

If the setup completed successfully, proceed to **1 - Neo4j Import** to load all data into Neo4j.